# Aegis Wave - Model Training & Validation
## Optimized for 2-Class Fall Detection on Heterogeneous Edge Devices

This notebook adapts the **Aegis Wave** methodology for real-world CSI data, handling common dataset quirks:
1. **Mismatched subcarriers**: Standardizes both 232-subcarrier (HP) and 64-subcarrier (ESP32) inputs to a 64x500 window.
2. **Data Robustness**: Skips malformed HDF5 files and handles subjects with missing classes.
3. **Split Consistency**: Adheres to the pre-defined ID-based splits (train/val/test).
4. **Edge Ready**: Outputs a quantized TFLite model with high-confidence thresholds.

In [11]:
import numpy as np
import pandas as pd
import h5py
import os
import json
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from scipy import signal
import matplotlib.pyplot as plt
from datetime import datetime

# Set constants
DATA_DIR = "data/"
SUB_COUNT = 64  # Standardized for both device types
TIME_STEPS = 500
EMERGENCY_THRESHOLD = 0.85
ANOMALY_THRESHOLD = 0.60
LABEL_MAP = {"Fall": 0, "Nonfall": 1}
INV_LABEL_MAP = {0: "Fall", 1: "Nonfall"}

## 1. Data Loading & Robust Processing
We use a generator approach to handle large amounts of raw HDF5 data without exhausting memory.

In [12]:
def butterworth_filter(data, lowcut=0.5, highcut=10, fs=100, order=4):
    """Apply Butterworth bandpass filter to isolate human movement"""
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype="band")
    return signal.filtfilt(b, a, data, axis=1)

def load_csi_sample(file_path):
    """Robust HDF5 loading with shape standardization"""
    full_path = os.path.join(DATA_DIR, file_path.lstrip("./"))
    try:
        with h5py.File(full_path, "r") as f:
            data = f["CSI_amps"][:]
            data = data.squeeze() 
            if data.shape[0] >= SUB_COUNT:
                data = data[:SUB_COUNT, :]
            else:
                return None
            if data.shape[1] < TIME_STEPS:
                return None
            data = data[:, :TIME_STEPS]
            data = butterworth_filter(data)
            return data.T 
    except Exception:
        return None

def load_dataset_from_ids(ids_file, metadata_df):
    """Loads all valid samples belonging to a split ID set"""
    with open(os.path.join(DATA_DIR, "splits", ids_file), "r") as f:
        ids = json.load(f)
    X, y = [], []
    for sample_id in ids:
        row = metadata_df[metadata_df["id"] == sample_id]
        if row.empty: continue
        sample = load_csi_sample(row.iloc[0]["file_path"])
        if sample is not None:
            X.append(sample)
            y.append(LABEL_MAP[row.iloc[0]["label"]])
    return np.array(X), np.array(y)

metadata_df = pd.read_csv(os.path.join(DATA_DIR, "metadata/sample_metadata.csv"))
X_train, y_train = load_dataset_from_ids("train_id.json", metadata_df)
X_val, y_val = load_dataset_from_ids("val_id.json", metadata_df)
X_test, y_test = load_dataset_from_ids("test_id.json", metadata_df)
print(f"Loaded Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Loaded Train: 2847, Val: 630, Test: 625


In [13]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(X_val.shape[0], -1)).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test.reshape(X_test.shape[0], -1)).reshape(X_test.shape)
y_train_cat = keras.utils.to_categorical(y_train, 2)
y_val_cat = keras.utils.to_categorical(y_val, 2)
y_test_cat = keras.utils.to_categorical(y_test, 2)


In [14]:
model = keras.Sequential([
    keras.layers.Conv1D(32, 5, activation="relu", input_shape=(TIME_STEPS, SUB_COUNT)),
    keras.layers.MaxPooling1D(2),
    keras.layers.Conv1D(64, 3, activation="relu"),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(2, activation="softmax")
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

c:\amos\projects\aegis-dlweek\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 496, 32)        │        10,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 248, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 246, 64)        │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,770 (81.13 KB)

 Trainable params: 20,770 (81.13 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(X_train_scaled, y_train_cat, validation_data=(X_val_scaled, y_val_cat), epochs=20, batch_size=32)


Epoch 1/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6898 - loss: 0.6191 - val_accuracy: 0.6984 - val_loss: 0.5480
Epoch 2/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7580 - loss: 0.5002 - val_accuracy: 0.7270 - val_loss: 0.4988
Epoch 3/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7678 - loss: 0.4607 - val_accuracy: 0.7556 - val_loss: 0.4613
Epoch 4/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7917 - loss: 0.4292 - val_accuracy: 0.7698 - val_loss: 0.4694
Epoch 5/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8054 - loss: 0.4113 - val_accuracy: 0.7698 - val_loss: 0.4416
Epoch 6/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8184 - loss: 0.3950 - val_accuracy: 0.7762 - val_loss: 0.4646
Epoch 7/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8226 - loss: 0.3795 - val_accuracy: 0.7794 - val_loss: 0.5145
Epoch 8/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8226 - loss: 0.3721 - val_accuracy: 0.7683 - v

In [16]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('aegis_wave_final.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"✅ TFLite model size: {len(tflite_model) / 1024:.2f} KB")
print(f"Suitable for: Raspberry Pi, ESP32, edge routers")

INFO:tensorflow:Assets written to: C:\Users\amosl\AppData\Local\Temp\tmp2qxdnjyp\assets


INFO:tensorflow:Assets written to: C:\Users\amosl\AppData\Local\Temp\tmp2qxdnjyp\assets


Saved artifact at 'C:\Users\amosl\AppData\Local\Temp\tmp2qxdnjyp'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 500, 64), dtype=tf.float32, name='keras_tensor_8')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2385893550288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2385893542416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2382906922704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2385893541072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2385893541648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2385893540688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2385893547408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2385893548368: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ TFLite model size: 27.81 KB
Suitable for: Raspberry Pi, ESP32, edge routers
